# Ch11. (2유형) 데이터 분석 기초

In [1]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('../data/elec_train.csv')
test = pd.read_csv('../data/elec_test.csv')

# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

# 회귀: 연속적 숫자 값 예측(집값, 기온 등)
# 분류: 클래스(범주) 예측(이진분류, 다중분류)

# 데이터 분석 프로세스: 1)탐색적 데이터 분석 -> 2)데이터 전처리 -> 3)데이터셋 분할 -> 4)학습 -> 5)평가 -> 6)예측 및 결과 저장


In [2]:
# 데이터 유형 파악: 숫자형/범주형 등 파악해 이후 전처리 방향 결정
print(train.info())
print(test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5100 entries, 0 to 5099
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   건물코드    5100 non-null   object 
 1   기온      5100 non-null   float64
 2   강수량     1114 non-null   float64
 3   풍속      5099 non-null   float64
 4   습도      5099 non-null   float64
 5   전력소비량   5100 non-null   float64
dtypes: float64(5), object(1)
memory usage: 239.2+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630 entries, 0 to 629
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   건물코드    630 non-null    object 
 1   기온      630 non-null    float64
 2   강수량     630 non-null    float64
 3   풍속      630 non-null    float64
 4   습도      630 non-null    int64  
dtypes: float64(3), int64(1), object(1)
memory usage: 24.7+ KB
None


In [3]:
# 결측치 파악: 모델 성능 저하의 원인이 되므로 사전에 파악 필요
print(train.isnull().sum())
print(test.isnull().sum())

건물코드        0
기온          0
강수량      3986
풍속          1
습도          1
전력소비량       0
dtype: int64
건물코드    0
기온      0
강수량     0
풍속      0
습도      0
dtype: int64


In [4]:
# 범주형 변수 카테고리 파악(object 형태)
print(train['건물코드'].value_counts())
print(test['건물코드'].value_counts())

건물코드
CM    51
DR    51
CY    51
BS    51
BN    51
      ..
CZ    51
DK    51
BB    51
DO    51
DL    51
Name: count, Length: 100, dtype: int64
건물코드
BF    7
CV    7
AY    7
AG    7
DC    7
     ..
BW    6
BQ    6
AW    6
BU    6
AC    6
Name: count, Length: 100, dtype: int64


In [5]:
# X, Y 데이터 셋 분리: 독립변수 X와 종속변수 Y를 분리... 모델 학습엔 독립변수만 활용
X_train = train.drop(['전력소비량'], axis=1) # 컬럼 단위 axis=1
y = train['전력소비량']

print(X_train.shape, y.shape, test.shape)

(5100, 5) (5100,) (630, 5)


In [6]:
# 결측치 처리: 평균값/최빈값 대체 or 제거 등
X_train['강수량'] = X_train['강수량'].fillna(0)
X_train['풍속'] = X_train['풍속'].fillna(X_train['풍속'].mean())
X_train['습도'] = X_train['습도'].fillna(X_train['습도'].mode()[0])

print(X_train.isnull().sum())

건물코드    0
기온      0
강수량     0
풍속      0
습도      0
dtype: int64


In [7]:
# 수치형 변수 스케일링 - MinMaxScaling(0~1 사이로 정규화)/StandardScaling(평균 0, 표준편차 1로 정규화)
# 모델이 특정 변수에 치우치지 않도록 값을 일정 범위로 조정

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
num_columns = X_train.select_dtypes(exclude='object').columns # 수치형 변수들만 가져와서 적용
X_train[num_columns] = scaler.fit_transform(X_train[num_columns])
test[num_columns] = scaler.transform(test[num_columns]) # test 데이터에는 fit x(학습x)
print(X_train.head())

  건물코드        기온       강수량        풍속        습도
0   CM  0.791837  0.000000  0.308271  0.809524
1   DR  0.612245  0.000000  0.203008  0.988095
2   CY  0.355102  0.000000  0.180451  1.000000
3   BS  0.469388  0.007752  0.135338  0.940476
4   BN  0.861224  0.000000  0.082707  0.559524


In [8]:
# 범주형 변수 인코딩 - LabelEncoding(순서형)/OneHotEncoding(명목형, 중복 피하기 위해 다중 컬럼화)
# 모델이 이해할 수 있도록 문자 데이터를 숫자로 변환

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
X_train['건물코드'] = encoder.fit_transform(X_train['건물코드']) # 범주형 변수인 건물코드만
test['건물코드'] = encoder.transform(test['건물코드']) # fit 빼고 transform만
print(X_train['건물코드'])

0       64
1       95
2       76
3       44
4       39
        ..
5095    72
5096    14
5097    61
5098    19
5099    86
Name: 건물코드, Length: 5100, dtype: int64


In [9]:
# 학습, 검증 데이터 셋 분할: 과적합 방지 위해 데이터 분리 후 일반화 성능 평가

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2) # 학습데이터가 80, 테스트데이터가 20
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(4080, 5) (1020, 5) (4080,) (1020,)


In [10]:
# Randomforest 활용 모델 학습: 학습용 데이터로 모델 학습... 분류 or 회귀 따라 모델 달라질 수 있음

from sklearn.ensemble import RandomForestRegressor # 앙상블 기법
model = RandomForestRegressor()
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [11]:
# MSE, R2 Score 활용 평가: 예측 결과와 실제값의 차이를 수치로 나타내 모델 성능 확인
# 회귀: MSE, RMSE, R2 등
# 분류: F1-score, ROC_AUC 등

from sklearn.metrics import mean_squared_error, r2_score
y_pred = model.predict(X_val)
mse = mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(mse, r2)

763603.8626998534 0.9008305680574599


In [13]:
# test 데이터 예측 및 결과 저장 (pred 1개 컬럼): 최종 모델을 test 데이터에 적용해 예측값 생성 -> 제출양식에 맞게 저장
y_pred = model.predict(test)
result = pd.DataFrame(y_pred, columns=['pred'])
print(result)
result.to_csv('result.csv', index=False) # 주로 index 넣지 말라고 출제

          pred
0     861.1362
1    2800.3914
2    2797.7586
3    1714.6566
4     714.4716
..         ...
625  1226.6166
626   704.7252
627  1150.0430
628  2444.3430
629  4515.9660

[630 rows x 1 columns]
